In [ ]:
import datetime
import fabric.functions as fn
import logging

udf = fn.UserDataFunctions()

In [ ]:
# 1. Eliminar registro indicado
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def Eliminar(sqlDB: fn.FabricSqlConnection,
                                           idunico: int ) -> str:
    
    connection = sqlDB.connect()
    cursor = connection.cursor()

    
    if idunico:
        delete_query = """
            DELETE FROM [dbo].[f_mov_budget]
            WHERE [ID] = ?
        """
        cursor.execute(delete_query, (idunico,))
        if cursor.rowcount == 0:
            raise fn.UserThrownError("No se encontró el ID para eliminar.", {"IDUNICO": idunico})

    
    connection.commit()
    cursor.close()
    connection.close()

    return f"Eliminado correctamente el registro con Id {idunico}."

In [ ]:
# 2. Actualizar campo "Importe" del registro
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def actualizar_importe_presupuesto(sqlDB: fn.FabricSqlConnection,
                         idunico: int,
                         importepresupuesto: int) -> int:

    if not importepresupuesto :
        raise fn.UserThrownError("Todos los campos (id, importe) son obligatorios.", {
            "importe": importepresupuesto
        })

    connection = sqlDB.connect()
    cursor = connection.cursor()

    update_query = """
        UPDATE [dbo].[f_mov_budget]
        SET [Importe] = ?
        WHERE [ID] = ?
    """

    cursor.execute(update_query, (importepresupuesto, idunico))

    if cursor.rowcount == 0:
        raise fn.UserThrownError("No se encontró ningún registro con ese Id.", {"ID": idunico})

    connection.commit()
    cursor.close()
    connection.close()

    return f"Registro actualizado correctamente para Id {idunico}."

In [ ]:
# 3. Insertar nuevo registro
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def Insertar_registro(sqlDB: fn.FabricSqlConnection,
                       clienteid: int,
                       cuentacontable: int,
                       descripcioncuenta: str,
                       ejercicio: int,
                       fecha: str,
                       grupo: int,
                       importe: int,
                       mes: int,
                       productoid: int,
                       tipo:  str) -> str:               

    if not clienteid or not cuentacontable or not descripcioncuenta or not ejercicio or not fecha or not grupo or not importe or not mes or not productoid or not tipo:
        raise fn.UserThrownError("Todos los campos a rellenar son obligatorios.", {
            "ClienteID": clienteid,
            "CuentaContable": cuentacontable,
            "DescripcionCuenta": descripcioncuenta,
            "Ejercicio": ejercicio,
            "Fecha": fecha,
            "Grupo": grupo,
            "Importe": importe,
            "Mes": mes,
            "ProductoID": productoid,
            "Tipo": tipo,

        })
    
    insert_query = """
        INSERT INTO [dbo].[f_mov_budget] (
            [Fecha],
            [Ejercicio],
            [Mes],
            [Grupo],
            [Tipo],
            [CuentaContable],
            [DescripcionCuenta],
            [ClienteID],
            [ProductoID],
            [Importe]
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """

   # Establecer conexión y ejecutar la consulta
    connection = sqlDB.connect()
    cursor = connection.cursor()
    try:
        cursor.execute(insert_query, (fecha, ejercicio, mes, grupo, tipo, cuentacontable, descripcioncuenta, clienteid, productoid, importe))
        connection.commit()
    finally:
        connection.close()

    return "Se insertó el nuevo registro correctamente."

In [ ]:
# 4. Copiar un registro existente
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def Copiar_registro(sqlDB: fn.FabricSqlConnection, idunicoorigen: int) -> str:
    if not idunicoorigen:
        raise fn.UserThrownError(
            "El campo ID de origen es obligatorio.",
            {"ID": idunicoorigen}
        )

    connection = sqlDB.connect()
    cursor = connection.cursor()

    try:
        copy_query = """
            INSERT INTO [dbo].[f_mov_budget] (
                [Fecha],
                [Ejercicio],
                [Mes],
                [Grupo],
                [Tipo],
                [CuentaContable],
                [DescripcionCuenta],
                [ClienteID],
                [ProductoID],
                [Importe]
            )
            OUTPUT INSERTED.[ID]
            SELECT
                [Fecha],
                [Ejercicio],
                [Mes],
                [Grupo],
                [Tipo],
                [CuentaContable],
                [DescripcionCuenta],
                [ClienteID],
                [ProductoID],
                [Importe]
            FROM [dbo].[f_mov_budget]
            WHERE [ID] = ?
        """

        cursor.execute(copy_query, (idunicoorigen,))
        row = cursor.fetchone()

        if not row:
            raise fn.UserThrownError(
                "No se encontró el registro origen para copiar.",
                {"ID": idunicoorigen}
            )

        nuevo_id = row[0]
        connection.commit()

    finally:
        cursor.close()
        connection.close()

    return f"Registro copiado correctamente. ID origen: {idunicoorigen}. Nuevo ID: {nuevo_id}."